In [1]:
import mat4py
import ntpath
import sys
import json as j
import numpy as np
from scipy.spatial.transform import Rotation
import tf_keras

In [2]:
def quaternion_to_euler_angle_vectorized1(w, x, y, z, degrees=True):
    
    rotation = Rotation.from_quat([w,x,y,z])
    euler_angles = rotation.as_euler('zyx', degrees=degrees)
    
    # Adjust the signs if needed
    euler_angles[0] *= -1  # Adjust the sign of roll
    euler_angles[2] *= -1  # Adjust the sign of yaw
    
    for i in range(3):
        if degrees:
            if np.abs(euler_angles[i]) > 90:
                euler_angles[i] = -(180 - np.abs(euler_angles[i]))
        else:
            if np.abs(euler_angles[i]) > np.pi/2:
                euler_angles[i] = -(np.pi - np.abs(euler_angles[i]))

    return euler_angles
    

In [3]:
def JSON2Mat(json):
    output_file = ntpath.basename(json).split(".")[0] + '_Processed' + '.mat'
    print("preparing: "+output_file)
    json_data = j.load(open(json))


    Skeletons = {}
    Skeletons["NrOfSegments"] = len(json_data["Skeletons"][0]["Segments"])
    Skeletons["NrOfRecords"] = len(json_data["Skeletons"][0]["Segments"][0]["Parts"][0]["Values"])
    for x in json_data["Skeletons"][0]["Segments"]:
        Skeletons[x["Name"]] = {}
        Skeletons[x["Name"]]["Local"] = "1"
        Skeletons[x["Name"]]["Position"] = []
        Skeletons[x["Name"]]["Quaternion"] = []
        Skeletons[x["Name"]]["Euler"] = []
        for y in x["Parts"][0]["Values"]:
            res = quaternion_to_euler_angle_vectorized1(y[6], y[3], y[4], y[5])
            temp = []
            temp.append(res[0])
            temp.append(res[1])
            temp.append(res[2])
            Skeletons[x["Name"]]["Euler"].append(temp) #(Roll,Pitch,Yaw)
            temp = []
            temp.append(y[6])
            temp.append(y[3])
            temp.append(y[4])
            temp.append(y[5])
            Skeletons[x["Name"]]["Quaternion"].append(temp) #(w,x,y,z)
            temp = []
            temp.append(y[0])
            temp.append(y[1])
            temp.append(y[2]) 
            Skeletons[x["Name"]]["Position"].append(temp) #(X,Y,Z)


    Analog = {}
    Analog["NrOfEMGSensors"] = len(json_data["Analog"][0]["Values"])

    i = 0
    for i in range(len(json_data["Analog"][0]["Values"])):
        name = str(json_data["Analog"][0]["Labels"][i]).replace(" ", "_")
        Analog[name] = []
        Analog[name].append(json_data["Analog"][0]["Values"][i])
        Analog[name] = np.transpose(Analog[name]).tolist()
        np.reshape(Analog[name], -1)

    data = {}
    data["Data_Container"] = {}
    data["Data_Container"]["Name"] = json_data["Name"]
    data["Data_Container"]["Skeletons"] = Skeletons
    data["Data_Container"]["Analog"] = Analog


    mat4py.savemat(output_file, data)


In [12]:
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_index_S.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_middle_S.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_ring_S.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_pinky_S.json")
JSON2Mat("Dynamic 28.json")

#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_index_F.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_middle_F.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_ring_F.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_pinky_F.json")
#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_thumb_F.json")

#JSON2Mat("Dataset_Raw\Subject2\Trial 4\JSON\HO_random.json")
print("done")

preparing: Dynamic 28_Processed.mat
done
